# Pricing engine — price and Greeks vs spot (European vs American)

Visual demonstration of the pricing engine. Percent `# %%` cells: open as a notebook or run as
a script. It ONLY calls the tested pricing engine and plots; no pricing logic lives here.

Run: `uv run --group notebooks python notebooks/demo_pricing_greeks.py`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from algotrading.core.config import load_config
from algotrading.infra.pricing import OptionStyle, PricingInput, PricingParams, price

Load the versioned pricing config (lattice steps, finite-difference bumps, unit conventions).
A kernel has no ``__file__``, so locate the repo root by walking up to the workspace marker.

In [ ]:
def _repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "packages").is_dir():
            return candidate
    raise FileNotFoundError("repo root (with pyproject.toml + packages/) not found")


_CONFIG = _repo_root() / "packages" / "infra" / "configs" / "pricing.yaml"
params = PricingParams.from_config(load_config(_CONFIG))

spots = np.linspace(70.0, 130.0, 60)


def _curve(style: OptionStyle, attr: str) -> np.ndarray:
    results = [
        price(
            PricingInput(
                spot=float(s),
                strike=100.0,
                maturity=0.5,
                vol=0.25,
                rate=0.04,
                carry=0.0,
                is_call=True,
                multiplier=100,
                style=style,
            ),
            params,
        )
        for s in spots
    ]
    return np.array([getattr(r, attr) for r in results])

Price and the first/second-order Greeks across spot, European (analytic) vs American (lattice).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for attr, ax in zip(["price", "delta", "gamma", "vega"], axes.flat, strict=True):
    ax.plot(spots, _curve(OptionStyle.EUROPEAN, attr), color="C0", label="European")
    ax.plot(spots, _curve(OptionStyle.AMERICAN, attr), color="C1", ls="--", label="American")
    ax.axvline(100.0, color="grey", lw=0.8, ls=":")  # strike
    ax.set(xlabel="spot", ylabel=attr, title=f"{attr} vs spot")
    ax.legend()
fig.tight_layout()
plt.show()